# Real F1 historical model test
Executed analysis of a sealed run. The three learned families are linear/logistic, random forest, and histogram gradient boosting. Simple baselines remain comparison references.

Training implementation: `src/f1ml_real/lab.py`. Reproduction and interpretation: `docs/real-model-lab.md`. Run this notebook from the repository root or `notebooks/`.

No raw FIA PDFs are redistributed here; the dataset retains public document URLs and SHA-256 hashes.


In [1]:
from pathlib import Path
import json
import pandas as pd
from f1ml.storage import verify_directory
from f1ml_real.lab import RealDataset, load_run
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
DATASET = ROOT / 'data/real-core-20261105-v4'
RUN = ROOT / 'artifacts/real-lab/replay-2026-19-all-bdc1c440786e'
source = RealDataset(DATASET)
saved = load_run(RUN)
source_receipt = verify_directory(DATASET)
run_receipt = verify_directory(RUN)
print('Source:', source_receipt['source'])
print('Dataset ID:', source_receipt['id'])
print('Run ID:', run_receipt['id'])
print('Evaluation:', source_receipt['mode'])
print(source_receipt['exposure'])

Source: FIA final classifications and independent entry lists
Dataset ID: 08bd7449688fda332cab42379d89cdeb295c3b200c45d2d9119e3aefc6befb2a
Run ID: fff62af8b7101a0ddcadc7254fdb146bc2987dfe0d3ae9c26e285859b3aca11d
Evaluation: historical_reconstruction_exploratory
Historical outcomes were previously accessible and consulted; no untouched holdout claim


In [2]:
events = pd.read_parquet(DATASET / 'events.parquet')
rosters = pd.read_parquet(DATASET / 'rosters.parquet')
labels = pd.read_parquet(DATASET / 'labels.parquet')
coverage = labels.groupby(['event_id', 'task']).agg(
    entrants=('driver_id', 'size'),
    known_numeric_ranks=('official_position', 'count'),
    observed_outcomes=('label_observed', 'sum'))
print(coverage.to_string())
print('Excluded feature groups:', ', '.join(source_receipt['missing_groups']))
print('Availability:', source_receipt['availability_semantics'])

                     entrants  known_numeric_ranks  observed_outcomes
event_id task                                                        
2026-01  qualifying        22                   19                 19
         race              22                   16                 22
2026-02  qualifying        22                   22                 22
         race              22                   15                 22
2026-03  qualifying        22                   22                 22
         race              22                   20                 22
2026-04  qualifying        22                   21                 22
         race              22                   18                 22
2026-05  qualifying        22                   22                 22
         race              22                   16                 22
2026-06  qualifying        22                   22                 22
         race              22                   16                 22
2026-07  qualifying 

In [3]:
config = json.loads((RUN / 'config.json').read_text())
print(json.dumps(config, indent=2, sort_keys=True))
assert (RUN / 'prediction_lock.json').exists()
print('Configuration and forecasts were sealed before target scoring.')

{
  "calibration": {
    "podium/gradient_boosting": {
      "coefficient": 1.3823111020485244,
      "event_ids": [
        "2026-17",
        "2026-18"
      ],
      "intercept": 0.9997286213352395,
      "method": "sigmoid"
    },
    "podium/linear": {
      "coefficient": 3.0328723995294564,
      "event_ids": [
        "2026-17",
        "2026-18"
      ],
      "intercept": 1.2882113662687342,
      "method": "sigmoid"
    },
    "podium/random_forest": {
      "coefficient": 2.3267450157563743,
      "event_ids": [
        "2026-17",
        "2026-18"
      ],
      "intercept": 0.6728467244328677,
      "method": "sigmoid"
    }
  },
  "calibration_event_ids": [
    "2026-17",
    "2026-18"
  ],
  "development_event_ids": [
    "2026-14",
    "2026-15",
    "2026-16"
  ],
  "excluded_features": {
    "podium/gradient_boosting": {
      "driver_qualifying_podium_rate_3": "entirely_missing_in_training",
      "driver_qualifying_podium_rate_5": "entirely_missing_in_training",
  

In [4]:
development = pd.read_parquet(RUN / 'development_metrics.parquet')
print('DEVELOPMENT: selection and calibration-fit stages')
print(development.to_string(index=False))
metrics = pd.read_parquet(RUN / 'metrics.parquet')
print('SELECTED HISTORICAL TARGET')
print(metrics.to_string(index=False))
print('Development and selected historical-target metrics are separate rows.')
print('Rank metrics use printed ranks; NC/DSQ tail ranks stay unknown.')

DEVELOPMENT: selection and calibration-fit stages
      task                         family      metric    value  event_count  known_label_count  total_driver_count  coverage                                  stage
qualifying                         linear         mae 2.812121            3                 64                  66  0.969697                  development_selection
qualifying                         linear        rmse 3.694011            3                 64                  66  0.969697                  development_selection
qualifying                         linear    spearman 0.816207            3                 64                  66  0.969697                  development_selection
qualifying                         linear top3_recall 0.444444            3                 64                  66  0.969697                  development_selection
qualifying                  random_forest         mae 2.812121            3                 64                  66  0.969697      

In [5]:
predictions = pd.read_parquet(RUN / 'predictions.parquet')
print(predictions.to_string(index=False))
print('Consulted historical outcomes: exploratory replay; no untouched holdout.')

event_id      driver_id            driver      team_id abbreviation       task            family  predicted_position  predicted_rank  raw_podium_probability  podium_probability  podium_prediction  threshold  actual_rank  actual_podium        outcome        label_available_at                                                                                                         label_source
 2026-19          albon   Alexander Albon     williams          ALB qualifying            linear           17.420455            20.0                     NaN                 NaN                NaN        NaN         16.0            NaN        ordered 2026-11-01T01:10:00+00:00 https://www.fia.com/system/files/decision-document/2026_mexico_city_grand_prix_-_final_qualifying_classification.pdf
 2026-19         alonso   Fernando Alonso aston_martin          ALO qualifying            linear           16.569706            18.0                     NaN                 NaN                NaN        NaN        